<a href="https://colab.research.google.com/github/mostafa-ja/LLM_from_scratch/blob/main/notebooks/00_start_here.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Start here: build a GPT-2 style LLM from scratch

This repository is a **learning project**. It builds a GPT-2 style language model and a byte-level BPE tokenizer from scratch, one concept at a time.

The goal is not to hide the model behind a framework. The goal is to make every important idea visible, runnable, and small enough to understand.

> Read the notebooks in order. Change the code. Break it. Fix it. That is the point.


## What you are building

```text
raw text
   |
   |  BPE tokenizer          (notebook 01 + src/tokenizer.py)
   v
token ids
   |
   |  token embedding + position embedding
   v
vector per token
   |
   |  N x Transformer block  (attention -> residual -> MLP -> residual)
   v
contextual vectors
   |
   |  LayerNorm + linear LM head
   v
logits over the vocabulary
   |
   |  cross-entropy during training, sampling during inference
   v
next token
```

Every box above is implemented in this repository. Nothing important is imported from a model library.


## Learning path

| # | Notebook | Core question |
|---|----------|---------------|
| 00 | **Start here** | How should I use this repository? |
| 01 | [Tokenizer from scratch](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/01_tokenizer_from_scratch.ipynb) | How does text become token ids? |
| 02 | [Bigram to self-attention](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/02_bigram_to_self_attention.ipynb) | How do tokens exchange information? |
| 03 | [GPT-2 architecture and optimizations](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/03_gpt2_architecture_and_optimizations.ipynb) | How is the full model assembled and trained? |
| 04 | [Train GPT-2 on Tiny Shakespeare](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/04_train_gpt2_on_tiny_shakespeare.ipynb) | What does a real training loop look like? |
| 05 | [Data pipeline](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/05_data_pipeline_text.ipynb) | How do I turn a text file into batches? |
| 06 | [FineWeb-Edu streaming](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/06_fineweb_edu_streaming.ipynb) | How do I prepare billions of tokens? |
| 07 | [Guided tour of the reference package](https://github.com/mostafa-ja/LLM_from_scratch/blob/main/notebooks/07_guided_tour.ipynb) | How do all the pieces fit together? |

The `practice/` folder contains scratch work. It is intentionally messy: it shows the learning process rather than a polished result.


## How to use this repository

1. **Read the notebook first.** Each notebook explains one layer of the stack with small experiments.
2. **Run the cells.** Change hyper-parameters, print shapes, plot losses, and inspect tensors.
3. **Read the matching Python module.** The `src/llm_from_scratch/` package is the same logic with type hints, docstrings, and tests.
4. **Do the exercises.** The fastest way to learn is to predict what will happen, run it, and compare.


In [ ]:
#@title Clone the repository and install the package
import os

REPO = "LLM_from_scratch"
if os.path.basename(os.getcwd()) != REPO:
    if not os.path.isdir(REPO):
        !git clone https://github.com/mostafa-ja/LLM_from_scratch.git
    os.chdir(REPO)

!pip install -q -e .

import llm_from_scratch
print("Package version:", llm_from_scratch.__version__)
print("Working directory:", os.getcwd())


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## The five ideas behind GPT

1. **Next-token prediction is the training signal.** Given a sequence, the model learns to predict the next token at every position. No labels are needed; the text labels itself.
2. **Attention mixes information across positions.** Each token builds a query, a key, and a value. A query is compared with previous keys to decide which earlier tokens matter.
3. **Residual connections and LayerNorm make depth trainable.** Each block adds its output back to its input. LayerNorm keeps activations at a stable scale.
4. **Cross-entropy turns probabilities into a loss.** The model outputs one score per vocabulary item. Softmax converts scores to probabilities, and cross-entropy punishes probability mass placed on the wrong next token.
5. **Sampling turns probabilities into text.** At inference time we sample from the next-token distribution instead of always taking the argmax.


## The reference package

The `src/llm_from_scratch/` package is a cleaned-up version of the notebooks:

| Module | What it contains |
|--------|------------------|
| `config.py` | GPT-2 size presets and configuration validation |
| `model.py` | Causal self-attention, MLP, Transformer block, GPT, generation |
| `tokenizer.py` | Byte-level BPE trainer with save/load support |
| `data.py` | Token datasets, a sequential loader, and memory-mapped shards |
| `train.py` | AdamW, gradient accumulation, cosine schedule, evaluation, checkpoints |
| `generate.py` | Load a checkpoint and sample text |
| `tests/` | Executable checks for shapes, causality, round-trips, and training |

Read the notebook first, then the module. The code is deliberately short enough to read in one sitting.


## This is not a black box

- The model is plain PyTorch: `nn.Module`, `nn.Linear`, `nn.Embedding`, `nn.LayerNorm`.
- The tokenizer is trained in front of you; there is no hidden vocabulary file.
- The training loop is explicit: forward, backward, clip, step, evaluate.
- The tests are small enough to read and modify.

If you want to understand a line, open the module and change it. Run `pytest` afterwards.


## Suggested exercises

- Train the BPE tokenizer with `vocab_size=256` and `vocab_size=1024`. Compare the number of tokens and the compression ratio.
- Change `block_size` in the model. Which tensors change shape? Which do not?
- Remove weight tying in `model.py`. How many parameters are added?
- Replace `F.scaled_dot_product_attention` with the manual attention formula from notebook 02. Do the outputs match?
- Train the nano model longer. Does validation loss keep falling, or does it start to overfit?
- Sample with `temperature=0.5`, `1.0`, and `1.5`. What changes?
- Run the FineWeb-Edu pipeline on a small token budget and inspect the shard files.


## Scope and honesty

- Notebooks 03 and 04 target the GPT-2 124M configuration and show the full training recipe. A full 124M pretraining run needs substantially more compute than a free Colab session.
- The small runs in this repository are demonstrations of the pipeline, not claims of state-of-the-art quality.
- The FineWeb-Edu notebook and script can produce multi-billion-token shards, but actually training on them requires a serious GPU budget.
- The code is written for clarity and experimentation. It is not a serving framework.

Being explicit about scope is part of good engineering.


## Acknowledgements

This project follows the spirit and structure of Andrej Karpathy's excellent tutorials:

- [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE)
- [Let's build GPT: from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY)
- [Let's reproduce GPT-2 (124M)](https://www.youtube.com/watch?v=l8pRSuU81PU)

The notebooks are the learning log. The `src/` package is the cleaned reference implementation.
